In [ ]:
# Notebook ini mengonversi jumlah penyewaan menjadi kategori diskrit (Low, Medium, High Demand) 
# untuk memprediksi tingkat keramaian di waktu tertentu.

# Proses ini menggunakan algoritma Decision Tree atau K-NN guna menghasilkan model 
# yang membantu perusahaan mengantisipasi stok sepeda secara efektif.

In [5]:
import sys
!{sys.executable} -m pip install scikit-learn feature-engine

  Obtaining dependency information for scikit-learn from https://files.pythonhosted.org/packages/9f/c4/0ab22726a04ede56f689476b760f98f8f46607caecff993017ac1b64aa5d/scikit_learn-1.8.0-cp312-cp312-win_amd64.whl.metadata
  Using cached scikit_learn-1.8.0-cp312-cp312-win_amd64.whl.metadata (11 kB)
  Obtaining dependency information for feature-engine from https://files.pythonhosted.org/packages/b1/04/06f8f92663e98f000d427a3e2ce20a2bcdc0119e30b4f97b918ee0f80fe4/feature_engine-1.9.3-py3-none-any.whl.metadata
  Using cached feature_engine-1.9.3-py3-none-any.whl.metadata (10 kB)
  Obtaining dependency information for scipy>=1.10.0 from https://files.pythonhosted.org/packages/c2/7f/acbd28c97e990b421af7d6d6cd416358c9c293fc958b8529e0bd5d2a2a19/scipy-1.16.3-cp312-cp312-win_amd64.whl.metadata
  Using cached scipy-1.16.3-cp312-cp312-win_amd64.whl.metadata (60 kB)
  Obtaining dependency information for joblib>=1.3.0 from https://files.pythonhosted.org/packages/7b/91/984aca2ec129e2757d1e4e3c81c3fcda9d

ERROR: Could not install packages due to an OSError: [WinError 32] The process cannot access the file because it is being used by another process: 'c:\\Users\\REYHAN\\AppData\\Local\\Programs\\Python\\Python312\\Lib\\site-packages\\scipy\\integrate\\_ivp\\__init__.py'
Consider using the `--user` option or check the permissions.


[notice] A new release of pip is available: 23.2.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [64]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
# Library sesuai proposal
from feature_engine.encoding import OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, confusion_matrix

In [54]:
# 1. Load Data
df = pd.read_csv('../data/hour.csv')

# Membuat salinan dataset
df_class = df.copy()
print(f"✓ Berhasil membuat salinan dataset. Jumlah baris: {df_class.shape[0]}")

✓ Berhasil membuat salinan dataset. Jumlah baris: 17379


In [55]:
# 2. Konversi 'cnt' menjadi Kelas Diskrit (Target) [cite: 58, 87]
def categorize_demand(cnt):
    if cnt < 50:
        return 'Low Demand'
    elif 50 <= cnt < 200:
        return 'Medium Demand'
    else:
        return 'High Demand' # Sesuai logika cnt >= 200

df_class['demand_category'] = df_class['cnt'].apply(categorize_demand)
print("✓ Pelabelan selesai. Contoh 5 baris pertama dengan kategori:")
display(df_class[['cnt', 'demand_category']].head())

✓ Pelabelan selesai. Contoh 5 baris pertama dengan kategori:


,cnt,demand_category
0,16,Low Demand
1,40,Low Demand
2,32,Low Demand
3,13,Low Demand
4,1,Low Demand


In [ ]:
# 3. Definisi Fitur dan Target [cite: 60]
# Kita masukkan variabel kategorikal untuk diproses oleh feature-engine, tapi sblm itu mengubah 
# data yang integer jadi object supaya terbaca
df_class[['season', 'weathersit']] = df_class[['season', 'weathersit']].astype(object)
features = ['season', 'hr', 'holiday', 'workingday', 'weathersit', 'temp', 'hum', 'windspeed']
X = df_class[features]
y = df_class['demand_category']

In [56]:
# 4. Feature Engineering menggunakan Feature-Engine [cite: 74, 75, 77]
# Menggunakan OneHotEncoder untuk variabel kategorikal sesuai proposal 
encoder = OneHotEncoder(variables=['season', 'weathersit'])
X_encoded = encoder.fit_transform(X)
print(f"✓ Feature Engineering selesai. Jumlah fitur setelah encoding: {X_encoded.shape[1]}")

✓ Feature Engineering selesai. Jumlah fitur setelah encoding: 14


In [ ]:
# 5. Split Data (Training & Testing)
X_train, X_test, y_train, y_test = train_test_split(X_encoded, y, test_size=0.2, random_state=42)

In [50]:
# 6. Pelatihan Model Decision Tree 
model_dt = DecisionTreeClassifier(random_state=42)
model_dt.fit(X_train, y_train)

,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.",'gini'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""best"" to choose the best split and ""random"" to choosethe best random split.",'best'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: int, float or {""sqrt"", ""log2""}, default=NoneThe number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... note:: The search for a split does not stop until at least one valid partition of the node samples is found, even if it requires to effectively inspect more than ``max_features`` features.",None
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for several splits and onesplit has to be selected at random. To obtain a deterministic behaviourduring fitting, ``random_state`` has to be fixed to an integer.See :term:`Glossary ` for details.",42
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow a tree with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current

In [60]:
# 7. SIMULASI INPUT DATASET BARU & PREDIKSI
# Sesuai proposal: "Pengguna diminta memasukkan dataset baru"
# Kita simulasikan dengan mengambil data dari X_test sebagai 'Data Baru'
print("\n5. Menyimulasikan input dataset baru...")
new_data_input = X_test.copy()

# Melakukan prediksi pada data baru
predictions = model_dt.predict(new_data_input)
new_data_input['Predicted_Demand_Category'] = predictions

print(f"   Berhasil memprediksi {len(new_data_input)} entri data baru.")


5. Menyimulasikan input dataset baru...
   Berhasil memprediksi 3476 entri data baru.


In [65]:
# 8. EKSPOR HASIL KE CSV (FOLDER BARU)
# Membuat folder baru jika belum ada
output_dir = '../data/hasil_klasifikasi'
if not os.path.exists(output_dir):
    os.makedirs(output_dir)
    print(f"6. Membuat folder baru: {output_dir}")

output_path = f"{output_dir}/classification_results.csv"
new_data_input.to_csv(output_path, index=False)

print(f"7. HASIL AKHIR: File berhasil disimpan di -> {output_path}")
print("-" * 50)
display(new_data_input[['Predicted_Demand_Category']].head(10))

6. Membuat folder baru: ../data/hasil_klasifikasi
7. HASIL AKHIR: File berhasil disimpan di -> ../data/hasil_klasifikasi/classification_results.csv
--------------------------------------------------


,Predicted_Demand_Category
12830,High Demand
8688,Medium Demand
7091,Low Demand
12230,High Demand
431,Low Demand
1086,Low Demand
11605,High Demand
7983,Low Demand
10391,Low Demand
7046,Low Demand


In [66]:
# 9. EVALUASI (OPSIONAL UNTUK PEMBUKTIAN)
print("\nEvaluasi Performa Model pada Data Test:")
print(classification_report(y_test, predictions))


Evaluasi Performa Model pada Data Test:
               precision    recall  f1-score   support

  High Demand       0.79      0.81      0.80      1267
   Low Demand       0.87      0.88      0.88       948
Medium Demand       0.72      0.70      0.71      1261

     accuracy                           0.79      3476
    macro avg       0.79      0.79      0.79      3476
 weighted avg       0.79      0.79      0.79      3476

